# Phase 4 — Pooling for query types 3 and 4

Run **BM25** and **TF-IDF** on semantic (type 3) and relational (type 4) queries, merge the top-k results into a **judging pool**, enrich rows from `documents.jsonl` (**caption**, **schema**, lexical **`text_preview`**, and dense-oriented **`embedding_text_preview`**), and export **Excel** for manual relevance labels.

**Retrieval** still uses only **`tokens`** (BM25) and **`text_blob`** (TF-IDF), matching [`scripts/build_index.py`](../../scripts/build_index.py). The extra **`embedding_text`** column is for **human review** only (same string used later for dense embeddings).

**After labeling:** fill `relevant` (e.g. `1` / `0` or `yes` / `no`), then convert to `data/qrels/qrels_type_3.json` and `qrels_type_4.json` (separate step / script).

**Requirements:** indices built on the **same** corpus you pool against — e.g. `python scripts/build_index.py --docs data/json_format_data/full/documents.jsonl` — so `models/bm25/` and `models/tfidf/` align with **`data/json_format_data/full/documents.jsonl`** (fallback: `subset_100k`).

In [1]:
# Setup — paths and imports
import json
import os
import re
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 30)


def find_root() -> Path:
    marker = Path("data") / "raw_data"
    for key in ("IR_PROJECT_ROOT", "VSCODE_WORKSPACE_FOLDER", "CURSOR_WORKSPACE_FOLDER"):
        v = os.environ.get(key)
        if v and (Path(v) / marker).exists():
            return Path(v)
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).exists():
            return p
    nb = globals().get("__vsc_ipynb_file__") or os.environ.get("JPY_SESSION_NAME", "")
    if nb:
        for p in Path(nb).parents:
            if (p / marker).exists():
                return p
    raise FileNotFoundError("Cannot find project root. Set IR_PROJECT_ROOT.")


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

QUERIES_DIR = ROOT / "data" / "queries"
MODELS_DIR = ROOT / "models"
DOCS_FULL = ROOT / "data" / "json_format_data" / "full" / "documents.jsonl"
DOCS_SUB = ROOT / "data" / "json_format_data" / "subset_100k" / "documents.jsonl"
DOCS_PATH = DOCS_FULL if DOCS_FULL.exists() else DOCS_SUB
OUT_DIR = ROOT / "data" / "pooling"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT      : {ROOT}")
print(f"Queries   : {QUERIES_DIR.exists()}")
print(f"BM25      : {(MODELS_DIR / 'bm25' / 'index.pkl').exists()}")
print(f"TF-IDF    : {(MODELS_DIR / 'tfidf' / 'matrix.npz').exists()}")
print(f"Documents : {DOCS_PATH}  exists={DOCS_PATH.exists()}")
if DOCS_FULL.exists():
    print(f"  (using full corpus for enrichment — align indices with build_index on this file)")
elif DOCS_SUB.exists():
    print(f"  (full corpus missing — using subset_100k; rebuild indices on subset if needed)")
print(f"Output    : {OUT_DIR}")


ROOT      : /Users/alireza/Library/CloudStorage/GoogleDrive-ali.esterabi@gmail.com/My Drive/QMUL_temr_2/IR_project
Queries   : True
BM25      : True
TF-IDF    : True
Documents : /Users/alireza/Library/CloudStorage/GoogleDrive-ali.esterabi@gmail.com/My Drive/QMUL_temr_2/IR_project/data/json_format_data/full/documents.jsonl  exists=True
  (using full corpus for enrichment — align indices with build_index on this file)
Output    : /Users/alireza/Library/CloudStorage/GoogleDrive-ali.esterabi@gmail.com/My Drive/QMUL_temr_2/IR_project/data/pooling


## Load retrievers

Loads pickled indices from `models/` (RAM-heavy: ~3–5 GB for BM25 corpus stats + sparse matrix for TF-IDF).

In [2]:
from src.retrieval.classical_ir import BM25Retriever, TFIDFRetriever

bm25 = BM25Retriever()
bm25.load(MODELS_DIR / "bm25")

tfidf = TFIDFRetriever()
tfidf.load(MODELS_DIR / "tfidf")


[BM25] Loaded index with 1,245,931 documents


[TFIDF] Loaded matrix (1245931, 398075) with 1,245,931 documents


## Pooling helpers

- **BM25 query tokens:** word tokens with the same boundary rule as the TF-IDF vectoriser (`\b\w+\b`), lowercased.
- **Pool:** union of top-`k` from each system per query; wide columns for rank/score and empty `relevant` for judging.

In [3]:
TOKEN_PATTERN = re.compile(r"(?u)\b\w+\b")


def query_tokens_bm25(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(text.lower())


def load_queries_json(path: Path) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        payload = json.load(f)
    return payload["queries"]


def pool_one_query(
    bm25: BM25Retriever,
    tfidf: TFIDFRetriever,
    query_id: str,
    query_type: int,
    primary_text: str,
    notes: str,
    k: int,
) -> list[dict]:
    b_hits = bm25.search(query_tokens_bm25(primary_text), k=k)
    t_hits = tfidf.search(primary_text, k=k)

    b_rank = {doc_id: r + 1 for r, (doc_id, _) in enumerate(b_hits)}
    b_score = dict(b_hits)
    t_rank = {doc_id: r + 1 for r, (doc_id, _) in enumerate(t_hits)}
    t_score = dict(t_hits)

    all_ids = list(dict.fromkeys(list(b_rank.keys()) + list(t_rank.keys())))

    rows = []
    for did in all_ids:
        rows.append(
            {
                "query_id": query_id,
                "query_type": query_type,
                "query_text": primary_text,
                "query_notes": notes,
                "doc_id": did,
                "bm25_rank": b_rank.get(did, ""),
                "bm25_score": b_score.get(did, ""),
                "tfidf_rank": t_rank.get(did, ""),
                "tfidf_score": t_score.get(did, ""),
                "in_bm25": int(did in b_rank),
                "in_tfidf": int(did in t_rank),
                "relevant": "",
                "judge_notes": "",
            }
        )
    return rows


def build_pool_dataframe(
    bm25: BM25Retriever,
    tfidf: TFIDFRetriever,
    queries: list[dict],
    k: int = 100,
) -> pd.DataFrame:
    all_rows: list[dict] = []
    for q in queries:
        qid = q["query_id"]
        qtype = int(q["query_type"])
        texts = q.get("query_texts") or []
        primary = texts[0] if texts else ""
        notes = q.get("notes", "")
        if not primary.strip():
            continue
        all_rows.extend(
            pool_one_query(bm25, tfidf, qid, qtype, primary, notes, k)
        )
    return pd.DataFrame(all_rows)


def enrich_captions(df: pd.DataFrame, docs_path: Path) -> pd.DataFrame:
    """Single pass over documents.jsonl: caption, schema, text_blob preview, embedding_text preview."""
    need = set(df["doc_id"].astype(str))
    meta: dict[str, dict] = {}
    if not docs_path.exists():
        print(f"Warning: {docs_path} not found — skipping enrichment")
        df = df.copy()
        df["caption"] = ""
        df["schema"] = ""
        df["text_preview"] = ""
        df["embedding_text_preview"] = ""
        return df

    print(f"Enriching {len(need):,} unique doc_ids from {docs_path.name} …")
    found = 0
    with open(docs_path, encoding="utf-8") as f:
        for line in f:
            doc = json.loads(line)
            did = doc["doc_id"]
            if did not in need:
                continue
            blob = doc.get("text_blob") or ""
            emb = doc.get("embedding_text") or ""
            meta[did] = {
                "caption": doc.get("caption", ""),
                "schema": doc.get("schema", ""),
                "text_preview": blob[:500] + ("…" if len(blob) > 500 else ""),
                "embedding_text_preview": emb[:500] + ("…" if len(emb) > 500 else ""),
            }
            found += 1
            if found >= len(need):
                break

    out = df.copy()
    out["caption"] = out["doc_id"].map(lambda x: meta.get(x, {}).get("caption", ""))
    out["schema"] = out["doc_id"].map(lambda x: meta.get(x, {}).get("schema", ""))
    out["text_preview"] = out["doc_id"].map(
        lambda x: meta.get(x, {}).get("text_preview", "")
    )
    out["embedding_text_preview"] = out["doc_id"].map(
        lambda x: meta.get(x, {}).get("embedding_text_preview", "")
    )
    missing = need - set(meta.keys())
    if missing:
        print(f"Warning: {len(missing)} doc_ids not found in corpus")
    return out


## Run pooling and export Excel

Adjust **`K_PER_SYSTEM`** (depth per retriever) and re-run. Output files:

- `data/pooling/pool_type_3.xlsx`
- `data/pooling/pool_type_4.xlsx`

Sheets include **`embedding_text_preview`** (dense-oriented prose from preprocessing) alongside **`text_preview`** (lexical `text_blob`).

In [4]:
K_PER_SYSTEM = 100

for qtype, fname in ((3, "queries_type_3.json"), (4, "queries_type_4.json")):
    qpath = QUERIES_DIR / fname
    if not qpath.exists():
        print(f"Skip (missing): {qpath}")
        continue
    queries = load_queries_json(qpath)
    df = build_pool_dataframe(bm25, tfidf, queries, k=K_PER_SYSTEM)
    df = enrich_captions(df, DOCS_PATH)
    out_xlsx = OUT_DIR / f"pool_type_{qtype}.xlsx"
    df.to_excel(out_xlsx, index=False, sheet_name="Pool")
    print(f"Wrote {out_xlsx.name}: {len(df):,} rows, {df['query_id'].nunique()} queries")

df.head(8)


Enriching 2,063 unique doc_ids from documents.jsonl …


Wrote pool_type_3.xlsx: 2,668 rows, 18 queries


Enriching 1,515 unique doc_ids from documents.jsonl …


Wrote pool_type_4.xlsx: 2,106 rows, 14 queries


,query_id,query_type,query_text,query_notes,doc_id,bm25_rank,bm25_score,tfidf_rank,tfidf_score,in_bm25,in_tfidf,relevant,judge_notes,caption,schema,text_preview,embedding_text_preview
0,Q4_001,4,What vessels does Sovcomflot own or manage?,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT,oc-companies-gb-16957183,1,19.320394,7,0.217491,1,1,,,AUREX INVEST LIMITED,Company,aurex invest limited other letting and operating of own or leased real estate private limited co...,AUREX INVEST LIMITED. Type: Company. Country: United Kingdom. Listed in: Ext Gb Coh Psc; Ann Gra...
1,Q4_001,4,What vessels does Sovcomflot own or manage?,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT,NK-3N7XijMsKL3q4ErA9mFkX9,2,17.280746,1,0.327259,1,1,,,STEVEN L VESSELS,Person,steven vessels steven vessels chiropractic chiropractic pract chiropractor debarment upin depart...,STEVEN L VESSELS. Type: Person. Country: United States. Sanctioned under: U.S. HHS Office of Ins...
2,Q4_001,4,What vessels does Sovcomflot own or manage?,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT,ru-7e51ad60ff407f550cb0afca4bf42a7d7d064c59,3,17.039417,2,0.288662,1,1,,,COVENANT VESSELS CORP.,Organization,covenant vessels corp ковенант весселс корп sanction linked,COVENANT VESSELS CORP.. Type: Organization. Listed in: Ext Ru Egrul; Ann Graph Topics. Registrat...
3,Q4_001,4,What vessels does Sovcomflot own or manage?,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT,NK-mMpoRMZUct3UnaiUnMdtdX,4,16.292267,3,0.281102,1,1,,,Scf Sakhalin Vessels Ltd,Company,scf sakhalin vessels ltd poi sanction linked гур мо україни,Scf Sakhalin Vessels Ltd. Type: Company. Country: Russian Federation. Sanctioned under: UA WS MA...
4,Q4_001,4,What vessels does Sovcomflot own or manage?,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT,NK-9QydunTAcdadkPnPTUdYqF,5,16.292267,4,0.279862,1,1,,,Scf Sakhalin Support Vessels,Company,scf sakhalin support vessels poi sanction linked гур мо україни,Scf Sakhalin Support Vessels. Type: Company. Country: Singapore. Sanctioned under: UA WS MARE. L...
5,Q4_001,4,What vessels does Sovcomflot own or manage?,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT,NK-CCLUNkU3ZyXeW6dFgAYECg,6,16.283155,26,0.160403,1,1,,,Atiya S A S,Company,atiya real estate activities with own or leased property sanction debarment office foreign asset...,Atiya S A S. Type: Company. Country: Colombia. Sanctioned under: U.S. Foreign Narcotics Kingpin ...
6,Q4_001,4,What vessels does Sovcomflot own or manage?,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT,NK-LJ2nJcQovs3g2wVXiuc8t2,7,16.283155,23,0.167922,1,1,,,Inmobiliaria Universal Deja Vu S.A. De C.V.,Company,inmobiliaria universal deja vu de real estate activities with own or leased property sanction de...,Inmobiliaria Universal Deja Vu S.A. De C.V.. Type: Company. Country: Mexico. Sanctioned under: U...
7,Q4_001,4,What vessels does Sovcomflot own or manage?,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT,NK-gSZELwQ6aMU34fTUz6Tu6x,8,16.03114,17,0.187658,1,1,,,"SKAIRU, S.A. DE C.V.",Company,skairu sa de cv skairu de real estate activities with own or leased property sanction debarment ...,"SKAIRU, S.A. DE C.V.. Type: Company. Country: Mexico. Sanctioned under: U.S. Foreign Narcotics K..."


## Convert manually-checked pooling results to qrels JSON

After manual grading in `pooling_results_manually_checked.xlsx` (sheet `Combined_Pooling_FINAL`):
- **Grade 2**: highly relevant
- **Grade 1**: somewhat relevant  
- **Grade 0**: not relevant → **excluded** from qrels

Outputs `data/qrels/qrels_type_3.json` and `qrels_type_4.json` with **graded relevance** (1/2) for nDCG@10 evaluation.

In [ ]:
from collections import defaultdict

# Path to manually-checked pooling results
POOLING_CHECKED = ROOT / "data" / "pooling" / "pooling_results_manually_checked.xlsx"
QRELS_OUT_DIR = ROOT / "data" / "qrels"
QRELS_OUT_DIR.mkdir(parents=True, exist_ok=True)

if not POOLING_CHECKED.exists():
    print(f"⚠️  {POOLING_CHECKED.name} not found — skip qrels generation")
else:
    print(f"Reading: {POOLING_CHECKED.name}")
    
    # Read manually-checked pooling results
    df_checked = pd.read_excel(
        POOLING_CHECKED, 
        sheet_name="Combined_Pooling_FINAL", 
        dtype=str
    ).fillna("")
    
    print(f"Loaded {len(df_checked):,} rows")
    print(f"Columns: {list(df_checked.columns)}")
    
    # Group by query type
    qrels_by_type = defaultdict(dict)
    stats = defaultdict(lambda: {"total": 0, "grade_1": 0, "grade_2": 0, "skipped": 0})
    
    for _, row in df_checked.iterrows():
        qid = str(row.get("query_id", "")).strip()
        if not qid:
            continue
        
        try:
            qtype = int(row.get("query_type", 0))
            doc_id = str(row.get("doc_id", "")).strip()
            grade_str = str(row.get("relevant", "")).strip()
            
            if not doc_id or not grade_str:
                continue
            
            grade = int(grade_str)
            
            # ✅ ONLY keep grade 1 and 2 (exclude 0 = not relevant)
            if grade == 0:
                stats[qtype]["skipped"] += 1
                continue
            
            if grade not in (1, 2):
                print(f"  Warning: Invalid grade {grade} for {qid}/{doc_id}, skipping")
                continue
            
            # Initialize query if needed
            if qid not in qrels_by_type[qtype]:
                qrels_by_type[qtype][qid] = {}
            
            # Add with graded relevance (1 or 2)
            qrels_by_type[qtype][qid][doc_id] = grade
            stats[qtype]["total"] += 1
            stats[qtype][f"grade_{grade}"] += 1
            
        except (ValueError, KeyError) as e:
            print(f"  Error processing row: {e}")
            continue
    
    # Write qrels JSON files
    print("\n" + "="*60)
    for qtype in sorted(qrels_by_type.keys()):
        qrels = qrels_by_type[qtype]
        out_path = QRELS_OUT_DIR / f"qrels_type_{qtype}.json"
        
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "query_type": qtype,
                    "n_queries": len(qrels),
                    "qrels": qrels,
                },
                f,
                indent=2,
                ensure_ascii=False,
            )
        
        # Print stats
        total_docs = sum(len(docs) for docs in qrels.values())
        s = stats[qtype]
        print(f"\n📊 Type {qtype}")
        print(f"  Queries         : {len(qrels)}")
        print(f"  Relevant docs   : {total_docs}")
        print(f"  Grade 1 (some)  : {s['grade_1']}")
        print(f"  Grade 2 (high)  : {s['grade_2']}")
        print(f"  Grade 0 (excl.) : {s['skipped']}")
        print(f"  ✅ Saved to     : {out_path.name}")
        
        # Sample
        if qrels:
            sample_qid = next(iter(qrels))
            sample_docs = list(qrels[sample_qid].items())[:3]
            print(f"  Sample [{sample_qid}]: {sample_docs}")
    
    print("\n" + "="*60)
    print("✅ Qrels generation complete!")